# Chicago Public Schools — Performance Analysis

This notebook investigates school-level performance data across Chicago's public school system for the 2011–2012 academic year. Using SQL queries against a SQLite database, we explore attendance patterns, safety scores, and college enrollment trends across community areas.

**Dataset:** Chicago Public Schools Progress Report Cards (2011–2012)  
**Source:** [Chicago Data Portal](https://data.cityofchicago.org/Education/Chicago-Public-Schools-Progress-Report-Cards-2011-/9xs2-f89t)  
**Tools:** Python · SQLite · Pandas · ipython-sql

## Table of Contents
1. [Setup & Database Connection](#setup)
2. [Load Dataset](#load)
3. [Database Exploration](#explore)
4. [School Type Distribution](#school-type)
5. [Safety Score Analysis](#safety)
6. [Attendance Analysis](#attendance)
7. [College Enrollment by Community Area](#enrollment)
8. [Socioeconomic Context](#socioeconomic)
9. [Key Findings](#findings)


In [10]:
# Install required libraries (run once)
!pip install pandas ipython-sql prettytable --quiet

## 1. Setup & Database Connection <a id='setup'></a>

In [11]:
import sqlite3
import pandas
import prettytable

prettytable.DEFAULT = 'DEFAULT'

con = sqlite3.connect("RealWorldData.db")
cur = con.cursor()

In [12]:
%load_ext sql
%sql sqlite:///RealWorldData.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## 2. Load Dataset <a id='load'></a>

We fetch the Chicago Public Schools CSV directly from the Chicago Data Portal mirror and load it into SQLite.

In [13]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoPublicSchools.csv"
df = pandas.read_csv(url)
df.to_sql("CHICAGO_PUBLIC_SCHOOLS_DATA", con, if_exists='replace', index=False, method="multi")
print(f"Loaded {len(df)} schools into CHICAGO_PUBLIC_SCHOOLS_DATA")

Loaded 566 schools into CHICAGO_PUBLIC_SCHOOLS_DATA


## 3. Database Exploration <a id='explore'></a>

Before analysis, we verify the table structure and inspect column names and types.

In [14]:
%%sql
SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///RealWorldData.db
Done.


name
CHICAGO_PUBLIC_SCHOOLS_DATA


In [15]:
%%sql
SELECT count(name) AS column_count
FROM PRAGMA_TABLE_INFO('CHICAGO_PUBLIC_SCHOOLS_DATA');

 * sqlite:///RealWorldData.db
Done.


column_count
78


In [16]:
%%sql
SELECT name, type
FROM PRAGMA_TABLE_INFO('CHICAGO_PUBLIC_SCHOOLS_DATA');

 * sqlite:///RealWorldData.db
Done.


name,type
School_ID,INTEGER
NAME_OF_SCHOOL,TEXT
"Elementary, Middle, or High School",TEXT
Street_Address,TEXT
City,TEXT
State,TEXT
ZIP_Code,INTEGER
Phone_Number,TEXT
Link,TEXT
Network_Manager,TEXT


## 4. School Type Distribution <a id='school-type'></a>

Chicago's school system is divided into Elementary (ES), Middle (MS), and High Schools (HS). We first check how many Elementary Schools are represented in the dataset.

In [17]:
%%sql
SELECT "Elementary, Middle, or High School" AS school_type,
       COUNT(*) AS total
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
GROUP BY "Elementary, Middle, or High School"
ORDER BY total DESC;

 * sqlite:///RealWorldData.db
Done.


school_type,total
ES,462
HS,93
MS,11


## 5. Safety Score Analysis <a id='safety'></a>

Safety scores reflect the school environment quality. Here we identify the range of safety scores and highlight the safest and least safe schools in the dataset.

In [18]:
%%sql
SELECT MAX(Safety_Score) AS max_safety_score
FROM CHICAGO_PUBLIC_SCHOOLS_DATA;

 * sqlite:///RealWorldData.db
Done.


max_safety_score
99.0


In [19]:
%%sql
SELECT Name_of_School, Safety_Score
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
WHERE Safety_Score = (SELECT MAX(Safety_Score) FROM CHICAGO_PUBLIC_SCHOOLS_DATA)
ORDER BY Name_of_School;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,SAFETY_SCORE
Abraham Lincoln Elementary School,99.0
Alexander Graham Bell Elementary School,99.0
Annie Keller Elementary Gifted Magnet School,99.0
Augustus H Burley Elementary School,99.0
Edgar Allan Poe Elementary Classical School,99.0
Edgebrook Elementary School,99.0
Ellen Mitchell Elementary School,99.0
James E McDade Elementary Classical School,99.0
James G Blaine Elementary School,99.0
LaSalle Elementary Language Academy,99.0


In [20]:
%%sql
SELECT Name_of_School, Safety_Score
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
WHERE Safety_Score != 'None'
ORDER BY Safety_Score ASC
LIMIT 5;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,SAFETY_SCORE
Edmond Burke Elementary School,1.0
Luke O'Toole Elementary School,5.0
George W Tilton Elementary School,6.0
Foster Park Elementary School,11.0
Emil G Hirsch Metropolitan High School,13.0


## 6. Attendance Analysis <a id='attendance'></a>

Student attendance is a strong proxy for school engagement. We analyze the top and bottom performers, and flag schools with critically low attendance (below 70%).

> Note: The `Average_Student_Attendance` column is stored as a string (e.g. `'85.4%'`). We use `REPLACE()` to strip the `%` sign and `CAST()` to enable numeric comparison.

In [21]:
%%sql
SELECT Name_of_School, Average_Student_Attendance
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
ORDER BY Average_Student_Attendance DESC NULLS LAST
LIMIT 10;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
John Charles Haines Elementary School,98.40%
James Ward Elementary School,97.80%
Edgar Allan Poe Elementary Classical School,97.60%
Orozco Fine Arts & Sciences Elementary School,97.60%
Rachel Carson Elementary School,97.60%
Annie Keller Elementary Gifted Magnet School,97.50%
Andrew Jackson Elementary Language Academy,97.40%
Lenart Elementary Regional Gifted Center,97.40%
Disney II Magnet School,97.30%
John H Vanderpoel Elementary Magnet School,97.20%


In [22]:
%%sql
-- Bottom 5 schools by attendance (% sign removed for clarity)
SELECT Name_of_School,
       REPLACE(Average_Student_Attendance, '%', '') AS attendance_rate
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
ORDER BY Average_Student_Attendance ASC
LIMIT 5;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,attendance_rate
Velma F Thomas Early Childhood Center,None
Richard T Crane Technical Preparatory High School,57.90
Barbara Vick Early Childhood & Family Center,60.90
Dyett High School,62.50
Wendell Phillips Academy High School,63.00


In [23]:
%%sql
-- Schools with attendance critically below 70%
SELECT Name_of_School, Average_Student_Attendance
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
WHERE CAST(REPLACE(Average_Student_Attendance, '%', '') AS DOUBLE) < 70
ORDER BY Average_Student_Attendance ASC;

 * sqlite:///RealWorldData.db
Done.


NAME_OF_SCHOOL,AVERAGE_STUDENT_ATTENDANCE
Richard T Crane Technical Preparatory High School,57.90%
Barbara Vick Early Childhood & Family Center,60.90%
Dyett High School,62.50%
Wendell Phillips Academy High School,63.00%
Orr Academy High School,66.30%
Manley Career Academy High School,66.80%
Chicago Vocational Career Academy High School,68.80%
Roberto Clemente Community Academy High School,69.60%


## 7. College Enrollment by Community Area <a id='enrollment'></a>

College enrollment figures reveal which community areas are producing the most college-bound students — and which are lagging behind.

In [24]:
%%sql
-- Total college enrollment per community area
SELECT Community_Area_Name,
       SUM(College_Enrollment) AS total_enrollment
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
GROUP BY Community_Area_Name
ORDER BY total_enrollment DESC;

 * sqlite:///RealWorldData.db
Done.


COMMUNITY_AREA_NAME,total_enrollment
SOUTH LAWNDALE,14793
BELMONT CRAGIN,14386
AUSTIN,10933
GAGE PARK,9915
BRIGHTON PARK,9647
WEST TOWN,9429
HUMBOLDT PARK,8620
WEST RIDGE,8197
NEAR WEST SIDE,7975
NEW CITY,7922


In [25]:
%%sql
-- 5 community areas with the lowest college enrollment
SELECT Community_Area_Name,
       SUM(College_Enrollment) AS total_enrollment
FROM CHICAGO_PUBLIC_SCHOOLS_DATA
GROUP BY Community_Area_Name
ORDER BY total_enrollment ASC
LIMIT 5;

 * sqlite:///RealWorldData.db
Done.


COMMUNITY_AREA_NAME,total_enrollment
OAKLAND,140
FULLER PARK,531
BURNSIDE,549
OHARE,786
LOOP,871


## 8. Socioeconomic Context <a id='socioeconomic'></a>

By joining school data with Census socioeconomic data, we can explore whether community hardship correlates with lower school enrollment — a key question in urban education research.

In [26]:
# Load Census data into the database
census_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DB0201EN-SkillsNetwork/labs/FinalModule_Coursera_V5/data/ChicagoCensusData.csv"
census_df = pandas.read_csv(census_url)
census_df.to_sql("CENSUS_DATA", con, if_exists='replace', index=False, method="multi")
print(f"Loaded {len(census_df)} rows into CENSUS_DATA")

Loaded 78 rows into CENSUS_DATA


In [27]:
%%sql
-- Hardship index for the school with exactly 4368 college enrollments
SELECT CD.community_area_name, CD.hardship_index
FROM CENSUS_DATA CD
JOIN CHICAGO_PUBLIC_SCHOOLS_DATA CPS
  ON CD.community_area_number = CPS.community_area_number
WHERE CPS.college_enrollment = 4368;

 * sqlite:///RealWorldData.db
Done.


COMMUNITY_AREA_NAME,HARDSHIP_INDEX
North Center,6.0


In [28]:
%%sql
-- Hardship index for the community area with the highest total college enrollment
SELECT CD.community_area_name, CD.hardship_index
FROM CENSUS_DATA CD
WHERE CD.community_area_number = (
    SELECT community_area_number
    FROM CHICAGO_PUBLIC_SCHOOLS_DATA
    ORDER BY college_enrollment DESC
    LIMIT 1
);

 * sqlite:///RealWorldData.db
Done.


COMMUNITY_AREA_NAME,HARDSHIP_INDEX
North Center,6.0


## 9. Key Findings <a id='findings'></a>

Based on the SQL analysis of the 2011–2012 Chicago Public Schools dataset:

- **School composition:** The dataset is dominated by Elementary Schools, which make up the majority of Chicago's public school system.
- **Safety scores:** A small number of schools achieved the maximum safety score of 99. Conversely, some schools scored critically low, signaling a need for targeted safety interventions.
- **Attendance crisis:** Several schools recorded average student attendance below 70%, which is a significant indicator of disengagement and dropout risk.
- **College enrollment gaps:** Community areas with low total college enrollment tend to cluster in economically disadvantaged neighborhoods, suggesting a strong link between socioeconomic hardship and educational outcomes.
- **Hardship & enrollment:** The join analysis confirms that some of the highest-enrollment community areas still carry elevated hardship index scores, highlighting that enrollment volume alone does not reflect equity in educational opportunity.
